# Exercises Feifei Monday 29.04

## Exercise 1. Population structure analysis using SNPs

**Dataset**

https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/1000_genomes_project/release/20181203_biallelic_SNV/

The SNV genotypes are made available in VCF files for each chromosome. You can start with SNV on chr1.

**Workflow**

VCF file -- PLINK convert -- LD Pruning -- PCA analysis -- ADMIXTURE Analysis

**Reading**

https://link.springer.com/protocol/10.1007/978-1-0716-0199-0_4 https://connor-french.github.io/intro-pop-structure-r/

### Answer

1. Install PLINK (v.1.9 or 2.0), Install ADMIXTURE
In Jupyter shell commands can be run with !



2. **VCF**:
    Use scikit-allel (Python equivalent of vcfR)

In [ ]:
# Prep: Installing packages into python environment (scikit-allel)

!pip install scikit-allel

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 14.7 MB/s  0:00:00


In [ ]:
# Prep: Installing packages into python environment (geopandas)
import sys
!{sys.executable} -m pip install geopandas

import geopandas as gpd
print(gpd.__version__)

   ---------------------------------------- 0.0/22.9 MB ? eta -:--:--
   --------- ------------------------------ 5.2/22.9 MB 25.5 MB/s eta 0:00:01
   ------------------- -------------------- 11.0/22.9 MB 25.1 MB/s eta 0:00:01
   ----------------------------- ---------- 16.8/22.9 MB 26.0 MB/s eta 0:00:01
   ---------------------------------------  22.8/22.9 MB 26.5 MB/s eta 0:00:01
   ---------------------------------------- 22.9/22.9 MB 23.6 MB/s  0:00:00
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ---------------------------------------  6.3/6.3 MB 33.6 MB/s eta 0:00:01
   ---------------------------------------- 6.3/6.3 MB 26.0 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 23.1 MB/s  0:00:00

   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
 

In [ ]:
# Prep: Install PLINK and ADMIXTURE

# With the following code in the anaconda prompt terminal

# conda install -c bioconda plink
# conda install -c bioconda admixture

# Check if download worked
!plink --version
!admixture --help

SyntaxError: invalid syntax (1064615331.py, line 3)

In [8]:
# Data handling
import pandas as pd
import numpy as np

# Genetic data (VCF)
import allel  # from scikit-allel
print(allel.__version__)

# Machine learning (PCA, clustering, DAPC-like)
from sklearn.decomposition import PCA, NMF
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cluster import KMeans

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Mapping
import geopandas as gpd

1.3.13


In [ ]:
# 1. Load/ Inspect VCF

# Using scikit-allel (a Python equivalent of vcfR)

vcf_file = "your_data.vcf"

callset = allel.read_vcf(vcf_file)
print(callset.keys())

In [ ]:
# 2. Convert VCF into PLINK format

# In this step, data is converted into a binary file (.bed, .bim, .fam)

!plink --vcf your_data.vcf --make-bed --out data_plink #shell command

In [ ]:
# 3. LD pruning

# In this step, correlated SNPs are removed. 
# Preparation for PCA and ADMIXTURE

!plink --bfile data_plink \
       --indep-pairwise 50 5 0.2 \
       --out pruned_data

# Then we extract pruned SNPs
!plink --bfile data_plink \
       --extract pruned_data.prune.in \
       --make-bed \
       --out data_pruned

# Parameters:
# 50 = window size (SNPs)
# 5 = step size
# 0.2 = LD threshold (r²)

In [ ]:
# 4. PCA analysis

!plink --bfile data_pruned \
       --pca 10 \
       --out pca_results

In [ ]:
# 5. ADMIXTURE analysis

# Run for different K values (number of populations)
!admixture data_pruned.bed 3
# in this case K=3

for K in 2 3 4 5; do admixture data_pruned.bed $K; done
#tries with multiple K

In [ ]:
# Plot ADMIXTURE results

import numpy as np

Q = np.loadtxt("data_pruned.3.Q")

plt.figure(figsize=(10,4))
plt.imshow(Q, aspect='auto')
plt.xlabel("Ancestry components")
plt.ylabel("Individuals")
plt.title("ADMIXTURE K=3")
plt.colorbar()
plt.show()

## Exercise 2. Population structure analysis using STRs

**Dataset**

https://drive.google.com/drive/folders/1fEy09eRa0Cs4O_paZvyO5rAwnfvdt7M-?usp=sharing 

The STR genotypes are available in CSV files for each chromosome. You can start with STR on one chromosome.

**Workflow**

CSV file -- Filtering -- PCA analysis -- Clustering analysis – Supervised classification

**Reading**

https://bmcbioinformatics.biomedcentral.com/articles/10.1186/s12859-024-05703-y